Problem 1: Trip duration
Part 1: Build a Regression Model¶
Build a regression to predict trip duration by using

Day of time
Distance between start and end stations (there might be more than one way to measure it)
Hour of day
Weekend indicator
Don't forget to model bias (this one is intentionally not used in lecture)
Also any thing you want to end
Part 2: Experiment Design
Ensure that you properly design your experiment to report unbiased performance metric you choose
Part 3 [Optional]: Visualize
Generate some fictional pickup and dropoff locations for bike trips (random pair selection)
Estimate trip duration for those say 10 trips
Visualize them on map using pydeck by using redish color for slower trips and greener for faster trips.

1. Mesafe hesabında kullandğım yöntemle kuş uçuşu mesafe hesaplamış oldum , yani bisikletlinin önünde hiçbir engel olmaması gerekiyor , ki öyle olmadığı için mesafe hesabım yanlış oluyor. Burada api üzerinden mesafe ölçme çözümünü bulabildim ancak ödevin amacının bu olmadığını düşündüğümden uygulamadım. Bunun yerine 1.2 gibi bir sayıyla çarparak mesafeyi biraz artırmaya çalıştım.
2. İkinci hücrede sürüş süresini 1 dakika ve 3 saat arasında filtreledim. Daha kısa ve uzun sürüşlerin problemli olduğunu varsaydım.

In [1]:
%%duckdb -o trip_feature

SELECT date_diff('second', start_at, stop_at) AS duration_sec,

    isodow(start_at) AS day_of_week,

    hour(start_at) AS hour,

    CASE 
        WHEN isodow(start_at) IN (6, 7) THEN 1 
        ELSE 0 
    END AS is_weekend,

    ST_Distance_Sphere(
        ST_Point("start station longitude", "start station latitude"),
        ST_Point("end station longitude", "end station latitude")
    )*1.2 AS distance_m,

FROM (
    SELECT 
        * EXCLUDE (tripduration, starttime, stoptime),

        -- timestamp parsing
        strptime(starttime, ['%m/%d/%Y %H:%M',
                              '%m/%d/%Y %H:%M:%S',
                              '%Y-%m-%d %H:%M:%S']) AS start_at,

        strptime(stoptime, ['%m/%d/%Y %H:%M',
                             '%m/%d/%Y %H:%M:%S',
                             '%Y-%m-%d %H:%M:%S']) AS stop_at

    FROM 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
)
ORDER BY random()
LIMIT 100000

,duration_sec,day_of_week,hour,is_weekend,distance_m
0,868,5,23,0,1630.827623
1,254,3,7,0,1091.827675
2,662,2,19,0,1547.244109
3,900,1,18,0,2482.290571
4,2833,5,19,0,390.918870
...,...,...,...,...,...
99995,219,6,18,1,570.527848
99996,951,4,15,0,2430.485804
99997,452,2,17,0,2150.205339
99998,812,6,13,1,913.583808


In [2]:
%%duckdb -o ml_table

SELECT *
FROM trip_feature

WHERE duration_sec BETWEEN 60 AND 10800
  AND distance_m > 0

  AND duration_sec IS NOT NULL
  AND distance_m IS NOT NULL
  AND day_of_week IS NOT NULL
  AND hour IS NOT NULL

,duration_sec,day_of_week,hour,is_weekend,distance_m
0,868,5,23,0,1630.827623
1,254,3,7,0,1091.827675
2,662,2,19,0,1547.244109
3,900,1,18,0,2482.290571
4,2833,5,19,0,390.918870
...,...,...,...,...,...
97749,219,6,18,1,570.527848
97750,951,4,15,0,2430.485804
97751,452,2,17,0,2150.205339
97752,812,6,13,1,913.583808


In [3]:
from sklearn.model_selection import train_test_split

X = ml_table.drop(columns=["duration_sec"])
y = ml_table["duration_sec"]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

In [4]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression().fit(X_train, y_train)
y_pred=reg.predict(X_val)

In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error ,root_mean_squared_error
import numpy as np

mse = mean_squared_error(y_val, y_pred)
mae = mean_absolute_error(y_val, y_pred)
rmse = root_mean_squared_error(y_val, y_pred)

print(f" MSE : {mse}")
print(f" MAE : {mae}")
print(f" RMSE : {rmse}")

#Çıkan sonuçları daha kolay yorumlayabilmek için y sürelerinin ortalamasını ekledim.
print(f" Y değerlerinin ortalaması : {np.mean(y_train)}")

 MSE : 369961.3772972907
 MAE : 358.9649602906547
 RMSE : 608.2445045352163
 Y değerlerinin ortalaması : 810.8667905612767


In [6]:
#Değerler iyi gelmediği için log'u ile devam ediyorum
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

reg.fit(X_train, y_train_log)
log_preds = reg.predict(X_val)
y_pred = np.expm1(log_preds)

from sklearn.metrics import mean_absolute_error, mean_squared_error
mae_log = mean_absolute_error(y_val, y_pred)
mse_log = mean_squared_error(y_val, y_pred)
rmse_log = root_mean_squared_error(y_val, y_pred)

print(f" Validation MSE : {mse_log}")
print(f" Validation MAE : {mae_log}")
print(f" Validation RMSE : {rmse_log}")
print(f" Y değerlerinin ortalaması : {np.mean(y_train)}")

#Model seçimi validation verisiyle yapıldıktan sonra son başarımı test verisinde ölçüyorum.
final_reg = LinearRegression().fit(X_train_val, np.log1p(y_train_val))
test_log_preds = final_reg.predict(X_test)
test_pred = np.expm1(test_log_preds)

test_mse = mean_squared_error(y_test, test_pred)
test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print(f" Test MSE : {test_mse}")
print(f" Test MAE : {test_mae}")
print(f" Test RMSE : {test_rmse}")

 Validation MSE : 448517.4049573622
 Validation MAE : 352.7199711506142
 Validation RMSE : 669.7144204490166
 Y değerlerinin ortalaması : 810.8667905612767
 Test MSE : 445122.01846319117
 Test MAE : 358.8075846262392
 Test RMSE : 667.1746536426509


In [7]:
import pandas as pd

def calculate_distance(
    start_point: tuple[float, float],
    end_point: tuple[float, float]
) -> float:
    start_lon, start_lat = start_point
    end_lon, end_lat = end_point

    result = duckdb.sql(
        """
        SELECT ST_Distance_Sphere(
            ST_Point(?, ?),
            ST_Point(?, ?)
        )
        """,
        params=[
            start_lon,
            start_lat,
            end_lon,
            end_lat
        ]
    ).fetchone()

    return float(result[0])

def generate_trips(
    start_points: list[tuple[float, float]],
    end_points: list[tuple[float, float]],
    dataset: pd.DataFrame,
    number_of_trips: int = 10
) -> pd.DataFrame:

    trips: list[dict[str, float | int]] = []

    for i in range(number_of_trips):
        start_point = start_points[i]
        end_point = end_points[i]

        distance_m = calculate_distance(
            start_point,
            end_point
        )

        random_row = dataset.sample(n=1).iloc[0]

        trip = {
            "start_lon": start_point[0],
            "start_lat": start_point[1],
            "end_lon": end_point[0],
            "end_lat": end_point[1],

            "day_of_week": int(random_row["day_of_week"]),
            "hour": int(random_row["hour"]),
            "is_weekend": int(random_row["is_weekend"]),

            "distance_m": float(distance_m)
        }

        trips.append(trip)

    return pd.DataFrame(trips)


#10 tane yolculuk oluşturdum.
trips_df = generate_trips(
start_points = [(-73.9772, 40.7527),(-73.9918, 40.7759),(-74.0048, 40.7198),(-73.9546, 40.7782),(-73.9897, 40.7359),
                (-74.0134, 40.7033),(-73.9496, 40.7962),(-73.9819, 40.7684),(-73.9365, 40.8047),(-74.0012, 40.7461)],
end_points = [(-73.9498, 40.7765),(-74.0062, 40.7352),(-73.9687, 40.7584),(-73.9875, 40.7448),(-73.9582, 40.7196),
              (-73.9821, 40.7298),(-73.9714, 40.7811),(-73.9443, 40.7529),(-73.9657, 40.7886),(-73.9764, 40.7107)],

#koordinatlar benzer çünkü bisikletle kat edilemeyecek mesafeler almak istemedim.
    
dataset=ml_table,
number_of_trips=10
)

feature_cols = ["day_of_week", "hour", "is_weekend", "distance_m"]

trips_df["predicted_duration"] = reg.predict(
    trips_df[feature_cols]
)

In [8]:
import pydeck as pdk

mean_duration = trips_df["predicted_duration"].mean()

trips_df["color"] = trips_df["predicted_duration"].apply(
    lambda x: [255, 0, 0] if x > mean_duration else [0, 255, 0]
)

layer = pdk.Layer(
    "ArcLayer",
    data=trips_df,
    get_source_position=["start_lon", "start_lat"],
    get_target_position=["end_lon", "end_lat"],
    get_source_color="color",
    get_target_color="color",
    get_width=5,
    pickable=True
)

view_state = pdk.ViewState(
    longitude=-73.98,
    latitude=40.75,
    zoom=10
)

trip_map = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    tooltip={
        "text": "Tahmini süre: {predicted_duration} dakika"
    }
)

trip_map

{
  "initialViewState": {
    "latitude": 40.75,
    "longitude": -73.98,
    "zoom": 10
  },
  "layers": [
    {
      "@@type": "ArcLayer",
      "data": [
        {
          "color": [
            0,
            255,
            0
          ],
          "day_of_week": 5,
          "distance_m": 3133.226014955163,
          "end_lat": 40.7765,
          "end_lon": -73.9498,
          "hour": 18,
          "is_weekend": 0,
          "predicted_duration": 6.976093856738096,
          "start_lat": 40.7527,
          "start_lon": -73.9772
        },
        {
          "color": [
            0,
            255,
            0
          ],
          "day_of_week": 1,
          "distance_m": 2029.813653371504,
          "end_lat": 40.7352,
          "end_lon": -74.0062,
          "hour": 18,
          "is_weekend": 0,
          "predicted_duration": 6.5812648512421426,
          "start_lat": 40.7759,
          "start_lon": -73.9918
        },
        {
          "color": [
            255,
            0,
            0
          ],
          "day_of_week": 2,
          "distance_m": 4185.117127170933,
          "end_lat": 40.7584,
          "end_lon": -73.9687,
          "hour": 7,
          "is_weekend": 0,
          "predicted_duration": 7.25714934996165,
          "start_lat": 40.7198,
          "start_lon": -74.0048
        },
        {
          "color": [
            255,
            0,
            0
          ],
          "day_of_week": 5,
          "distance_m": 3799.3283594938725,
          "end_lat": 40.7448,
          "end_lon": -73.9875,
          "hour": 8,
          "is_weekend": 0,
          "predicted_duration": 7.142019143325127,
          "start_lat": 40.7782,
          "start_lon": -73.9546
        },
        {
          "color": [
            255,
            0,
            0
          ],
          "day_of_week": 3,
          "distance_m": 3538.201059435749,
          "end_lat": 40.7196,
          "end_lon": -73.9582,
          "hour": 9,
          "is_weekend": 0,
          "predicted_duration": 7.050941741350667,
          "start_lat": 40.7359,
          "start_lon": -73.9897
        },
        {
          "color": [
            255,
            0,
            0
          ],
          "day_of_week": 2,
          "distance_m": 3573.9417300410573,
          "end_lat": 40.7298,
          "end_lon": -73.9821,
          "hour": 17,
          "is_weekend": 0,
          "predicted_duration": 7.110127632950823,
          "start_lat": 40.7033,
          "start_lon": -74.0134
        },
        {
          "color": [
            0,
            255,
            0
          ],
          "day_of_week": 7,
          "distance_m": 2468.0431237433304,
          "end_lat": 40.7811,
          "end_lon": -73.9714,
          "hour": 13,
          "is_weekend": 1,
          "predicted_duration": 6.850439674621137,
          "start_lat": 40.7962,
          "start_lon": -73.9496
        },
        {
          "color": [
            255,
            0,
            0
          ],
          "day_of_week": 7,
          "distance_m": 4207.953467453046,
          "end_lat": 40.7529,
          "end_lon": -73.9443,
          "hour": 16,
          "is_weekend": 1,
          "predicted_duration": 7.468240236948127,
          "start_lat": 40.7684,
          "start_lon": -73.9819
        },
        {
          "color": [
            0,
            255,
            0
          ],
          "day_of_week": 2,
          "distance_m": 3284.3960463038925,
          "end_lat": 40.7886,
          "end_lon": -73.9657,
          "hour": 14,
          "is_weekend": 0,
          "predicted_duration": 6.991482231324582,
          "start_lat": 40.8047,
          "start_lon": -73.9365
        },
        {
          "color": [
            0,
            255,
            0
          ],
          "day_of_week": 1,
          "distance_m": 2963.672973077467,
          "end_lat": 40.7107,
          "end_lon": -73.9764,
          "hour": 16,
          "is_weekend": 